# FieldShift — TensorFlow intent-classification experiment

This notebook trains a small **task classifier**, not a generative LLM. The production assistant already works through sourced retrieval. No training data or weights are fabricated. Running all cells without reviewed examples performs readiness checks and skips training.

Use a separate Python 3.11 environment. Suggested initial compatibility environment: tensorflow==2.16.2, tf-keras==2.16.0, numpy==1.26.4, notebook. These packages are not installed by the app and this training environment has not been executed here. Pin the resolved environment after testing. Do not install into the system Python 3.14 environment.

References: https://www.tensorflow.org/js/guide/conversion and https://www.tensorflow.org/js/guide/platform_environment .


In [ ]:
from pathlib import Path
import json, random, hashlib, importlib.util, sys
ROOT = Path.cwd() if (Path.cwd() / "server").exists() else Path.cwd().parent
DATA = ROOT / "notebooks/data/intent_examples.jsonl"
SEED = 42
TRAIN = False
print({"python": sys.version, "dataset_exists": DATA.exists(), "tensorflow_installed": importlib.util.find_spec("tensorflow") is not None})


## Reviewed data contract

Supply JSONL rows with text, label, group_id, source_url, license and reviewed=true. Labels: crop, water, soil, rotation, out_of_scope. group_id must identify a farmer/source/paraphrase group; whole groups go into one split. Have people verify labels and license, include unsupported questions and contrasting regions, and avoid this app's own template outputs as an evaluation target. At least 20 examples per class and 10 distinct groups are required before this experiment trains. This minimum is a guard, not evidence that the dataset is sufficient.


In [ ]:
LABELS = ["crop", "water", "soil", "rotation", "out_of_scope"]
rows = [json.loads(line) for line in DATA.read_text(encoding="utf-8").splitlines() if line.strip()] if DATA.exists() else []
for row in rows:
    assert all(row.get(key) for key in ["text", "label", "group_id", "source_url", "license"])
    assert row["reviewed"] is True and row["label"] in LABELS
groups = sorted({row["group_id"] for row in rows})
random.Random(SEED).shuffle(groups)
split_a, split_b = int(len(groups)*.7), int(len(groups)*.85)
train_groups, val_groups, test_groups = set(groups[:split_a]), set(groups[split_a:split_b]), set(groups[split_b:])
splits = {name: [r for r in rows if r["group_id"] in group] for name, group in [("train", train_groups), ("validation", val_groups), ("test", test_groups)]}
assert not (train_groups & val_groups or train_groups & test_groups or val_groups & test_groups)
ready = len(groups) >= 10 and all(sum(r["label"] == label for r in rows) >= 20 for label in LABELS) and all(splits.values())
print({"ready": ready, "counts": {name: len(values) for name, values in splits.items()}, "training_enabled": TRAIN})


## Train and evaluate only after review

The held-out test is never used for fitting or choosing settings. Record a keyword baseline, macro F1, per-class precision/recall, confusion matrix, abstentions and a manual failure review before accepting the model. No validation metric is claimed until execution. The export below is a Keras artifact, not a verified TF.js or WebLLM model.


In [ ]:
if TRAIN and ready:
    import numpy as np
    import tensorflow as tf
    tf.keras.utils.set_random_seed(SEED)
    def arrays(name):
        return np.asarray([r["text"] for r in splits[name]], dtype=str), np.asarray([LABELS.index(r["label"]) for r in splits[name]])
    x_train, y_train = arrays("train")
    x_val, y_val = arrays("validation")
    x_test, y_test = arrays("test")
    vectorizer = tf.keras.layers.TextVectorization(max_tokens=4000, output_sequence_length=48)
    vectorizer.adapt(x_train)
    model = tf.keras.Sequential([tf.keras.Input(shape=(), dtype=tf.string), vectorizer,
        tf.keras.layers.Embedding(4000, 32), tf.keras.layers.GlobalAveragePooling1D(), tf.keras.layers.Dense(len(LABELS), activation="softmax")])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=15, batch_size=16,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)])
    predicted = model.predict(x_test).argmax(axis=1)
    matrix = tf.math.confusion_matrix(y_test, predicted, num_classes=len(LABELS)).numpy()
    print("Held-out confusion matrix", LABELS, matrix, sep="\n")
    precision = np.diag(matrix) / np.maximum(matrix.sum(axis=0), 1)
    recall = np.diag(matrix) / np.maximum(matrix.sum(axis=1), 1)
    f1 = 2*precision*recall / np.maximum(precision+recall, 1e-9)
    print({"macro_f1": float(f1.mean()), "precision": precision.tolist(), "recall": recall.tolist()})
    output = ROOT / "artifacts/intent-experiment"
    output.mkdir(parents=True, exist_ok=True)
    model.save(output / "intent.keras")
    report = {"seed": SEED, "labels": LABELS, "dataset_sha256": hashlib.sha256(DATA.read_bytes()).hexdigest(), "groups": {"train": sorted(train_groups), "validation": sorted(val_groups), "test": sorted(test_groups)}, "macro_f1": float(f1.mean()), "confusion": matrix.tolist()}
    (output / "evaluation.json").write_text(json.dumps(report, indent=2))
else:
    print("SKIPPED: enable TRAIN only after supplying enough reviewed, licensed, group-split examples.")


## Export acceptance

Verify a supported TF.js conversion route for the installed TensorFlow/Keras versions. Export vocabulary and implement equivalent preprocessing if the browser runtime cannot handle TextVectorization. Compare saved-model and browser logits on held-out cases, record conversion commands and versions, and benchmark assets/memory/latency before app integration. This classifier must never author crop recommendations, NASA readings or citations.
